# 04 — Surcharge d'opérateurs

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre les **dunder methods** (`__xxx__`) comme hooks du langage
- implémenter `__eq__`, `__lt__` et `__hash__`
- surcharger les opérateurs arithmétiques (`__add__`, `__mul__`, …)
- utiliser `functools.total_ordering` pour n'écrire qu'un seul comparateur
- connaître les pièges : mutabilité vs hachabilité, `__hash__ = None`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, héritage, `super()`
- `@property`, `@classmethod`
- `isinstance`

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `@dataclass` qui génère tout ça automatiquement (jour 2)
- `Protocol` structurel (notebook 05)

## Plan

1. Qu'est-ce qu'un *dunder* ?
2. Égalité : `__eq__` et `__hash__`
3. Ordre : `__lt__`, `__le__`, …
4. `functools.total_ordering`
5. Arithmétique : `__add__`, `__sub__`, `__mul__`, …
6. Conversion : `__int__`, `__float__`, `__bool__`
7. Collections : `__len__`, `__getitem__`, `__contains__`
8. Pièges
9. Synthèse
10. Exercices

---

## 1. Qu'est-ce qu'un *dunder* ?

Les méthodes à double underscore (`__xxx__`, *dunder* pour *double underscore*) sont des **hooks** que Python appelle en réponse à une syntaxe. Quand vous écrivez `a + b`, Python appelle en réalité `a.__add__(b)`. Les surcharger donne à vos objets un comportement naturel dans le langage.

In [ ]:
(5).__add__(3)  # équivalent de 5 + 3


In [ ]:
'abc'.__add__('def')


---

## 2. Égalité : `__eq__` et `__hash__`

Par défaut, `a == b` compare les identités (adresses mémoire). Pour obtenir une égalité structurelle, on implémente `__eq__`.

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __eq__(self, autre: object) -> bool:
        if not isinstance(autre, Point):
            return NotImplemented
        return self.x == autre.x and self.y == autre.y

    def __repr__(self) -> str:
        return f'Point({self.x}, {self.y})'


In [ ]:
Point(1, 2) == Point(1, 2)


In [ ]:
Point(1, 2) == Point(1, 3)


In [ ]:
Point(1, 2) == 42  # NotImplemented → Python essaie à l'envers → False


### Et `__hash__` alors ?

Dès qu'on définit `__eq__`, **Python met `__hash__ = None`** automatiquement. L'objet devient **non hachable** : plus possible de le mettre dans un `set` ou comme clé de `dict`. C'est voulu : deux objets égaux doivent avoir le même hash, et Python n'a aucun moyen de deviner votre règle.

In [ ]:
try:
    {Point(1, 2)}
except TypeError as exc:
    print(exc)


### Rendre l'objet hachable

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    def __eq__(self, autre: object) -> bool:
        if not isinstance(autre, Point):
            return NotImplemented
        return (self.x, self.y) == (autre.x, autre.y)
    def __hash__(self) -> int:
        return hash((self.x, self.y))
    def __repr__(self) -> str:
        return f'Point({self.x}, {self.y})'


In [ ]:
{Point(1, 2), Point(1, 2), Point(3, 4)}


⚠️ **Un objet mutable ne devrait jamais être hachable**. Si vous modifiez `self.x`, le hash change, et l'objet est introuvable dans le `set` où il a été placé.

---

## 3. Ordre : `__lt__`, `__le__`, `__gt__`, `__ge__`

Pour pouvoir trier ou comparer avec `<`, `<=`, `>`, `>=`, il faut implémenter ces dunders.

In [ ]:
class Produit:
    def __init__(self, nom: str, prix: float) -> None:
        self.nom = nom
        self.prix = prix
    def __lt__(self, autre: 'Produit') -> bool:
        return self.prix < autre.prix
    def __repr__(self) -> str:
        return f'{self.nom}({self.prix})'


In [ ]:
produits = [Produit('pain', 1.2), Produit('vin', 12.0), Produit('lait', 0.9)]
sorted(produits)


Avec juste `__lt__`, `sorted()` fonctionne. Mais `a > b` ne marche pas, `a >= b` non plus.

---

## 4. `functools.total_ordering`

Décorateur de classe qui, à partir de `__eq__` et d'**un** seul opérateur d'ordre (`__lt__` par exemple), génère tous les autres.

In [ ]:
from functools import total_ordering

@total_ordering
class Produit:
    def __init__(self, nom: str, prix: float) -> None:
        self.nom = nom
        self.prix = prix
    def __eq__(self, autre: object) -> bool:
        if not isinstance(autre, Produit):
            return NotImplemented
        return self.prix == autre.prix
    def __lt__(self, autre: 'Produit') -> bool:
        return self.prix < autre.prix
    def __repr__(self) -> str:
        return f'{self.nom}({self.prix})'


In [ ]:
a = Produit('pain', 1.2); b = Produit('lait', 0.9)
a > b, a <= b, a >= b


---

## 5. Arithmétique : `__add__`, `__sub__`, `__mul__`, …

In [ ]:
class Vec2:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    def __add__(self, autre: 'Vec2') -> 'Vec2':
        return Vec2(self.x + autre.x, self.y + autre.y)
    def __sub__(self, autre: 'Vec2') -> 'Vec2':
        return Vec2(self.x - autre.x, self.y - autre.y)
    def __mul__(self, k: float) -> 'Vec2':
        return Vec2(self.x * k, self.y * k)
    __rmul__ = __mul__  # pour que `3 * v` marche
    def __repr__(self) -> str:
        return f'Vec2({self.x}, {self.y})'


In [ ]:
Vec2(1, 2) + Vec2(3, 4)


In [ ]:
Vec2(5, 6) - Vec2(1, 1)


In [ ]:
2 * Vec2(3, 4)


### `__rxxx__` — l'opérateur « à droite »

Quand Python évalue `a + b`, il essaie `a.__add__(b)`. Si c'est `NotImplemented` ou que `a` ne connaît pas `b`, il essaie `b.__radd__(a)`. D'où `__rmul__` ci-dessus : il gère `3 * v` (où `int` ne sait pas multiplier par un `Vec2`).

---

## 6. Conversion : `__int__`, `__float__`, `__bool__`

Ces dunders définissent comment votre objet se convertit vers un type primitif.

In [ ]:
class Panier:
    def __init__(self) -> None:
        self.articles: list[str] = []
    def ajouter(self, a: str) -> None:
        self.articles.append(a)
    def __len__(self) -> int:
        return len(self.articles)
    def __bool__(self) -> bool:
        return len(self) > 0


In [ ]:
p = Panier(); bool(p)


In [ ]:
p.ajouter('pain'); bool(p)


⚠️ Si vous ne définissez pas `__bool__`, Python retombe sur `__len__`. Un objet avec `len == 0` est falsy.

---

## 7. Collections : `__len__`, `__getitem__`, `__contains__`

Donner à votre classe l'API d'une séquence ou d'un mapping.

In [ ]:
class Playlist:
    def __init__(self, titres: list[str]) -> None:
        self._titres = titres
    def __len__(self) -> int:
        return len(self._titres)
    def __getitem__(self, i: int) -> str:
        return self._titres[i]
    def __contains__(self, t: str) -> bool:
        return t in self._titres


In [ ]:
pl = Playlist(['a', 'b', 'c'])


In [ ]:
len(pl), pl[1], 'a' in pl


In [ ]:
for t in pl: print(t)  # itérable gratis via __getitem__


---

## 8. Pièges

- **`__eq__` désactive `__hash__`** — redéfinir explicitement si l'objet est immuable.
- **`__eq__` doit retourner `NotImplemented`** (pas `False`) si les types sont incompatibles. Ça laisse la main à l'autre opérande.
- **Mutable + hachable = bombe à retardement.** Ne hachez que des objets immuables (ou avec un hash indépendant de l'état modifiable).
- **`__repr__` sans guillemets pour `self.x` numérique, avec `!r` pour les chaînes.**
- **Préférez `@dataclass`** (jour 2) qui génère `__eq__`, `__hash__`, `__repr__` à votre place.

---

## Synthèse

| Dunder | Opérateur | Rôle |
|---|---|---|
| `__eq__`, `__ne__` | `==`, `!=` | Égalité structurelle |
| `__lt__`, `__le__`, `__gt__`, `__ge__` | `<`, `<=`, `>`, `>=` | Ordre |
| `__hash__` | `hash(obj)` | Hachage (sets, dict keys) |
| `__add__`, `__sub__`, `__mul__`, … | `+`, `-`, `*`, … | Arithmétique |
| `__radd__`, `__rmul__`, … | | Opérateur avec `self` à droite |
| `__len__`, `__getitem__`, `__contains__` | `len`, `[]`, `in` | API de collection |
| `__int__`, `__float__`, `__bool__` | `int()`, `float()`, `bool()` | Conversion |


### Règles à retenir

1. **`__eq__` retourne `NotImplemented` sur type étranger**, jamais `False`.
2. **Définir `__eq__` sans `__hash__`** rend l'objet non hachable — c'est voulu.
3. **Un objet mutable ne doit pas être hachable.** Point.
4. **`functools.total_ordering`** évite d'écrire 6 dunders d'ordre quand un seul suffit.
5. **Avant de tout surcharger à la main**, regardez si `@dataclass` ne le fait pas pour vous.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Égalité structurelle *(facile)*

Écrire une classe `Couleur` avec `r`, `g`, `b` (entiers 0..255) et implémenter `__eq__` telle que `Couleur(255, 0, 0) == Couleur(255, 0, 0)` soit `True`. Rendre l'objet hachable.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Surcharge_operateurs", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class Couleur:
    def __init__(self, r: int, g: int, b: int) -> None:
        self.r, self.g, self.b = r, g, b
    def __eq__(self, autre: object) -> bool:
        if not isinstance(autre, Couleur):
            return NotImplemented
        return (self.r, self.g, self.b) == (autre.r, autre.g, autre.b)
    def __hash__(self) -> int:
        return hash((self.r, self.g, self.b))
    def __repr__(self) -> str:
        return f'Couleur({self.r}, {self.g}, {self.b})'

print(Couleur(255, 0, 0) == Couleur(255, 0, 0))
print({Couleur(0, 0, 0), Couleur(0, 0, 0)})
```

</details>

### Exercice 2 — Tri avec `total_ordering` *(moyen)*

Écrire `Livre(titre, pages)` avec `__eq__` (par titre) et `__lt__` (par nombre de pages), décoré de `@total_ordering`. Trier une liste de 4 livres.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Surcharge_operateurs", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from functools import total_ordering

@total_ordering
class Livre:
    def __init__(self, titre: str, pages: int) -> None:
        self.titre = titre
        self.pages = pages
    def __eq__(self, autre: object) -> bool:
        if not isinstance(autre, Livre):
            return NotImplemented
        return self.titre == autre.titre
    def __lt__(self, autre: 'Livre') -> bool:
        return self.pages < autre.pages
    def __repr__(self) -> str:
        return f'{self.titre}({self.pages}p)'

livres = [Livre('A', 300), Livre('B', 100), Livre('C', 500), Livre('D', 200)]
print(sorted(livres))
```

</details>

### Exercice 3 — Vecteur avec arithmétique complète *(moyen)*

Implémenter `Vec3(x, y, z)` avec `__add__`, `__sub__`, `__mul__` (par scalaire), `__rmul__`, `__neg__` (l'unaire `-v`), `__eq__` et `__repr__`. Vérifier `v + w`, `2 * v`, `-v`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Surcharge_operateurs", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Vec3:
    def __init__(self, x: float, y: float, z: float) -> None:
        self.x, self.y, self.z = x, y, z
    def __add__(self, a: 'Vec3') -> 'Vec3':
        return Vec3(self.x + a.x, self.y + a.y, self.z + a.z)
    def __sub__(self, a: 'Vec3') -> 'Vec3':
        return Vec3(self.x - a.x, self.y - a.y, self.z - a.z)
    def __mul__(self, k: float) -> 'Vec3':
        return Vec3(self.x * k, self.y * k, self.z * k)
    __rmul__ = __mul__
    def __neg__(self) -> 'Vec3':
        return Vec3(-self.x, -self.y, -self.z)
    def __eq__(self, a: object) -> bool:
        if not isinstance(a, Vec3):
            return NotImplemented
        return (self.x, self.y, self.z) == (a.x, a.y, a.z)
    def __repr__(self) -> str:
        return f'Vec3({self.x}, {self.y}, {self.z})'

v, w = Vec3(1, 2, 3), Vec3(4, 5, 6)
print(v + w, 2 * v, -v)
```

</details>

### Exercice 4 — Collection custom *(difficile)*

Écrire une classe `Inventaire` (fil rouge) qui enveloppe un `dict[str, int]` (nom → quantité). Implémenter `__len__`, `__getitem__` (par nom), `__contains__` (par nom), `__iter__` (itère sur les noms), et une méthode `ajouter(nom, qte)`. Vérifier qu'une `Inventaire` peut être itérée dans un `for`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Surcharge_operateurs", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class Inventaire:
    def __init__(self) -> None:
        self._stock: dict[str, int] = {}
    def ajouter(self, nom: str, qte: int) -> None:
        self._stock[nom] = self._stock.get(nom, 0) + qte
    def __len__(self) -> int:
        return len(self._stock)
    def __getitem__(self, nom: str) -> int:
        return self._stock[nom]
    def __contains__(self, nom: str) -> bool:
        return nom in self._stock
    def __iter__(self):
        return iter(self._stock)

inv = Inventaire()
inv.ajouter('clavier', 3)
inv.ajouter('souris', 5)
inv.ajouter('clavier', 2)
print(len(inv), inv['clavier'], 'souris' in inv)
for nom in inv:
    print(nom, inv[nom])
```

</details>

### Exercice 5 — Nombre rationnel *(difficile)*

Implémenter une classe `Rationnel(num, den)` avec `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__eq__`, `__repr__`. Réduire la fraction à chaque construction (avec `math.gcd`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Surcharge_operateurs", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import math

class Rationnel:
    def __init__(self, num: int, den: int) -> None:
        if den == 0:
            raise ValueError('dénominateur nul')
        g = math.gcd(num, den)
        if den < 0:
            num, den = -num, -den
        self.num, self.den = num // g, den // g
    def __add__(self, a: 'Rationnel') -> 'Rationnel':
        return Rationnel(self.num * a.den + a.num * self.den, self.den * a.den)
    def __sub__(self, a: 'Rationnel') -> 'Rationnel':
        return Rationnel(self.num * a.den - a.num * self.den, self.den * a.den)
    def __mul__(self, a: 'Rationnel') -> 'Rationnel':
        return Rationnel(self.num * a.num, self.den * a.den)
    def __truediv__(self, a: 'Rationnel') -> 'Rationnel':
        return Rationnel(self.num * a.den, self.den * a.num)
    def __eq__(self, a: object) -> bool:
        if not isinstance(a, Rationnel):
            return NotImplemented
        return (self.num, self.den) == (a.num, a.den)
    def __repr__(self) -> str:
        return f'{self.num}/{self.den}'

a = Rationnel(1, 2); b = Rationnel(1, 3)
print(a + b, a - b, a * b, a / b)
```

</details>

---

## Ressources externes

### Documentation officielle
- [Data model — special method names](https://docs.python.org/3/reference/datamodel.html#special-method-names)
- [`functools.total_ordering`](https://docs.python.org/3/library/functools.html#functools.total_ordering)

### Lectures complémentaires
- Fluent Python, chap. 13 *Operator Overloading*.